# 01_Theory Practice: BPE 및 WordPiece 밑바닥부터 구현하기
이 노트북은 `01_theory.md`에서 설명한 서브워드 토크나이제이션 알고리즘을 파이썬 기본 자료구조만 사용하여 직접 구현해 보는 실습입니다.

## 1. BPE (Byte Pair Encoding) 직접 구현하기
이론 문서에 등장한 `low`, `lowest`, `newer`, `wider` 예제를 활용하여 가장 자주 등장하는 글자 쌍이 병합되는 과정을 확인합니다.

In [ ]:
import re
from collections import defaultdict

# 초기 데이터: 단어와 빈도수 (끝에 </w> 추가)
vocab = {
    'l o w </w>': 5,
    'l o w e s t </w>': 2,
    'n e w e r </w>': 6,
    'w i d e r </w>': 3
}

def get_stats(vocab):
    """인접한 두 토큰(글자/서브워드)의 쌍(pair) 빈도를 계산합니다."""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    """가장 빈도가 높은 쌍을 하나의 토큰으로 병합합니다."""
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# 병합 시뮬레이션
num_merges = 3
print("==== 초기 상태 ====")
for word, freq in vocab.items():
    print(f"'{word}': {freq}")
print()

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    # 가장 빈도가 높은 쌍 찾기
    best_pair = max(pairs, key=pairs.get)
    print(f"[{i+1}회차 병합] 가장 빈도가 높은 쌍: {best_pair} (등장 횟수: {pairs[best_pair]})")
    
    # 병합 수행
    vocab = merge_vocab(best_pair, vocab)
    
    print("-> 병합 후 어휘 상태:")
    for word, freq in vocab.items():
        print(f"   '{word}': {freq}")
    print()


## 2. WordPiece Score (PMI) 계산해보기
단순 빈도가 아닌, 우도(Likelihood)와 PMI(점별 상호 정보량) 개념을 활용하는 WordPiece의 점수 계산 방식을 코드로 확인합니다.
공식: $Score = P(ab) / (P(a) \times P(b))$

In [ ]:
# 임의의 말뭉치 통계 가정
total_tokens = 10000

# 단일 토큰 등장 횟수
counts = {
    'play': 500,
    '##ing': 800,
    'a': 2000,
    'the': 3000
}

# 결합 토큰 등장 횟수
pair_counts = {
    ('play', '##ing'): 450,  # play 뒤에 ing가 매우 자주 옴
    ('a', 'the'): 10        # a 뒤에 the가 오는 경우는 드물거나 거의 없음
}

def calc_wordpiece_score(a, b):
    p_a = counts[a] / total_tokens
    p_b = counts[b] / total_tokens
    p_ab = pair_counts[(a, b)] / total_tokens
    
    # PMI 기반 Score 계산
    score = p_ab / (p_a * p_b)
    return score, p_a, p_b, p_ab

print("==== WordPiece Score 계산 비교 ====\n")

# 1. 의미적 연결이 강한 경우 (play + ##ing)
score1, p_a1, p_b1, p_ab1 = calc_wordpiece_score('play', '##ing')
print(f"[의미적 종속] 'play' + '##ing'")
print(f" - P(a): {p_a1:.4f}, P(b): {p_b1:.4f}, P(ab): {p_ab1:.4f}")
print(f" - Score: {score1:.2f}\n")

# 2. 의미적 연결이 없는 경우 (a + the)
score2, p_a2, p_b2, p_ab2 = calc_wordpiece_score('a', 'the')
print(f"[단순 독립 단어] 'a' + 'the'")
print(f" - P(a): {p_a2:.4f}, P(b): {p_b2:.4f}, P(ab): {p_ab2:.4f}")
print(f" - Score: {score2:.2f}\n")

print("""결과 해석:
단순 빈도로만 치면 'a'와 'the'가 훨씬 많이 등장하지만, 
WordPiece 방식(Score)에서는 'play'와 '##ing'의 점수가 압도적으로 높게 나옵니다. 
따라서 WordPiece는 'play##ing'을 우선적으로 병합 후보로 선택하게 됩니다.""")
